# 02 — Data Cleaning

**Purpose:** Stream the 6 raw Shona sources through the cleaning pipeline and produce `data/processed/corpus.txt`.

**Expected runtime:** ~2 minutes (Mac, single-threaded).

**Pipeline (cheap → expensive):**
1. Strip HTML, URLs, emails
2. Normalize whitespace + Unicode (NFC)
3. Sentence-split each raw paragraph
4. Per-sentence length filter (≥ 5 words)
5. Per-sentence non-Latin char-ratio filter (≤ 30%)
6. **Shona word-vocabulary detector** (Bible-trained, threshold 0.20) — calibrated to reject Kinyarwanda/Swahili/Zulu/Tswana
7. Exact dedup (Blake2b sentence hash)

**Outputs:**
- `data/processed/corpus.txt` — one cleaned Shona sentence per line
- `data/processed/corpus_stats.json` — per-source dropoff counters + total

**Why this language detector and not `langdetect`:** see [`src/data/cleaner.py:ShonaLangDetector`](../src/data/cleaner.py). Standard library detectors confuse Shona with Kinyarwanda (same Bantu-language failure that mislabeled mC4). Char-trigram detectors have the same problem. Word-level matching against the Bible's known-Shona vocabulary is the only approach that gave a clean separation.

In [1]:
import logging
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(name)s | %(message)s",
    force=True,
)
logger = logging.getLogger("02_data_cleaning")

In [2]:
from src.data.bible_downloader import load_config
from src.data.cleaner import (
    CleaningConfig, ShonaLangDetector, ShonaCorpusCleaner, build_default_sources,
)

cfg_yaml = load_config(PROJECT_ROOT / "configs/config.yaml")
clean_cfg = CleaningConfig.from_dict(cfg_yaml.get("cleaning", {}))
print(f"CleaningConfig: {clean_cfg}")

sources = build_default_sources(PROJECT_ROOT)
for name, path in sources.items():
    exists = path.exists()
    size_mb = path.stat().st_size / 1024 / 1024 if exists else 0
    status = "OK" if exists else "MISSING"
    print(f"  {name:18} {status:8} {size_mb:>8.1f} MB")

CleaningConfig: CleaningConfig(min_words_per_sentence=5, max_non_shona_char_ratio=0.3, min_shona_lang_score=0.2, exact_dedup=True)
  bible              OK            3.8 MB
  wikipedia          OK            8.5 MB
  masakhane_ner      OK            1.4 MB
  masakhane_news     OK            3.5 MB
  glotcc             OK           34.7 MB
  hplt               OK          184.3 MB


## Train the language detector on the Bible

The Bible is the high-purity anchor — you verified it's 100% Shona. Building the detector's vocabulary from it gives a domain-correct reference for the per-sentence language ID step.

**Don't add Wikipedia to the training vocab.** I tested this — Shona Wikipedia has enough English-text contamination (article references, infoboxes, technical loanwords) that adding it makes the detector accept English. Bible-only is the precise anchor.

In [3]:
detector = ShonaLangDetector.from_corpus(
    PROJECT_ROOT / "data/raw/bible/bible_verses.txt",
)

2026-05-24 14:26:13,371 | INFO | src.data.cleaner |   vocab += /Users/apple/Desktop/taurabot/data/raw/bible/bible_verses.txt: +63377 unique / +476702 occurrences


2026-05-24 14:26:13,381 | INFO | src.data.cleaner | Vocabulary: 63377 unique words (min_len=3, min_freq=1) from 1 sources


In [4]:
# Calibration sanity check — re-run the test cases on this notebook's detector
_cases = [
    ("Bible verse",       "Pakutanga Mwari akasika matenga nenyika."),
    ("Wikipedia Shona",   "Chirimba izita remhuri rinowanikwa muZimbabwe."),
    ("HPLT Shona",        "Chokwadi Pamusoro paBaba, Mwanakomana, uye Mudzimu Mutsvene"),
    ("Kinyarwanda",       "Mwaramutse neza mu rugendo rwawe rwa buri munsi. Imana iguhe umugisha mwiza."),
    ("Swahili",           "Habari ya asubuhi. Jina langu ni Mary na ninatoka Nairobi nchini Kenya."),
    ("English",           "Hello my name is John and I am from London."),
]
print(f"{'text type':18} {'score':>6}  ({'accept' if True else 'reject'} at thresh={clean_cfg.min_shona_lang_score})")
print("-" * 60)
for label, text in _cases:
    s = detector.score(text)
    verdict = '✓' if s >= clean_cfg.min_shona_lang_score else '✗'
    print(f"  {label:16} {s:>6.3f}  {verdict}")

text type           score  (accept at thresh=0.2)
------------------------------------------------------------
  Bible verse       1.000  ✓
  Wikipedia Shona   0.400  ✓
  HPLT Shona        0.857  ✓
  Kinyarwanda       0.091  ✗
  Swahili           0.000  ✗
  English           0.000  ✗


## Run the cleaning pipeline

Streams all 6 sources, writes survivors to `data/processed/corpus.txt`. The output file is written atomically (via `.tmp` rename) — a killed run leaves no half-written file behind.

In [5]:
import time

cleaner = ShonaCorpusCleaner(clean_cfg, detector, sources)

t0 = time.time()
result = cleaner.run(
    out_path=PROJECT_ROOT / "data/processed/corpus.txt",
    stats_path=PROJECT_ROOT / "data/processed/corpus_stats.json",
)
print(f"\nCleaned in {time.time()-t0:.1f}s — {result['total_sentences']:,} sentences, {result['total_chars']/1024/1024:.1f} MB")

2026-05-24 14:26:13,397 | INFO | src.data.cleaner | Cleaning [bible] from /Users/apple/Desktop/taurabot/data/raw/bible/bible_verses.txt


2026-05-24 14:26:15,162 | INFO | src.data.cleaner |   [bible] raw=31103 → length=38525 → char_ratio=38525 → lang_id=38525 → dedup=38308 (kept 38308 sentences)


2026-05-24 14:26:15,162 | INFO | src.data.cleaner | Cleaning [wikipedia] from /Users/apple/Desktop/taurabot/data/raw/wikipedia/wiki_text.txt


2026-05-24 14:26:19,271 | INFO | src.data.cleaner |   [wikipedia] raw=115575 → length=87454 → char_ratio=87429 → lang_id=71008 → dedup=58210 (kept 58210 sentences)


2026-05-24 14:26:19,271 | INFO | src.data.cleaner | Cleaning [masakhane_ner] from /Users/apple/Desktop/taurabot/data/raw/masakhane/ner_text.txt


...

2026-05-24 14:26:35,365 | INFO | src.data.cleaner | Cleaning [hplt] from /Users/apple/Desktop/taurabot/data/raw/hplt/hplt_text.txt


2026-05-24 14:28:03,267 | INFO | src.data.cleaner |   [hplt] raw=1201596 → length=1568101 → char_ratio=1567918 → lang_id=1475593 → dedup=1096968 (kept 1096968 sentences)


2026-05-24 14:28:03,274 | INFO | src.data.cleaner | Wrote 1372354 cleaned sentences (160.6 MB) to /Users/apple/Desktop/taurabot/data/processed/corpus.txt


2026-05-24 14:28:03,275 | INFO | src.data.cleaner | Stats written to /Users/apple/Desktop/taurabot/data/processed/corpus_stats.json



Cleaned in 110.0s — 1,372,354 sentences, 160.6 MB


## Per-source dropoff table

In [6]:
print(f"{'source':18} {'raw_lines':>10} {'+len':>10} {'+char':>10} {'+lang':>10} {'+dedup':>10}  {'kept':>10}")
print("-" * 92)
for src in result["per_source"]:
    print(f"{src['source']:18} {src['raw_lines']:>10,} {src['after_length']:>10,} "
          f"{src['after_char_ratio']:>10,} {src['after_lang_id']:>10,} {src['after_dedup']:>10,} "
          f"{src['sentences_kept']:>10,}")
print("-" * 92)
total = result['total_sentences']
print(f"{'TOTAL kept':80} {total:>10,}")

source              raw_lines       +len      +char      +lang     +dedup        kept
--------------------------------------------------------------------------------------------
bible                  31,103     38,525     38,525     38,525     38,308     38,308
wikipedia             115,575     87,454     87,429     71,008     58,210     58,210
masakhane_ner           8,867      8,756      8,756      8,558      8,555      8,555
masakhane_news          3,684     21,796     21,796     20,252     20,164     20,164
glotcc                298,401    253,563    253,358    230,628    150,149    150,149
hplt                1,201,596  1,568,101  1,567,918  1,475,593  1,096,968  1,096,968
--------------------------------------------------------------------------------------------
TOTAL kept                                                                        1,372,354


In [7]:
# Sample 10 random cleaned sentences to eyeball quality
import random
random.seed(7)

with (PROJECT_ROOT / "data/processed/corpus.txt").open(encoding="utf-8") as fh:
    lines = [ln.rstrip() for ln in fh]

print(f"Sampling 10 of {len(lines):,} cleaned sentences:\n")
for s in random.sample(lines, 10):
    print(f"  > {s[:140]}{'…' if len(s)>140 else ''}")

Sampling 10 of 1,372,354 cleaned sentences:

  > Baba vacho vakada mupanduki akanga adzoka kudarika zvavaimboita (vhesi 20).
  > Panyaya yefundo vanga vazvi simudzira.
  > - Vanhu mazana matanhatu vakasara vasina pekugara mushure mekukukurwa kwevhu. (Vanhu mazana matanhatu vakasiyiwa vasina pekugara mushure mek…
  > Basin inogona kushandiswa kana munhu mapurasitiki waya netting pasi, kunogona grubs, earthworm mumudziyo.
  > VaHwende vaudza vanhu vanga vachitsvaira muEpworth ava kuti bato ravo rinotarisira kuti makanzuru ose aite basa nemazvo sezvo makanzura ebat…
  > Kuti uwane kudzikiswa kwakanaka muzana muzana remafuta emuviri, zvinokurudzirwa kuita maminetsi makumi matatu ecardio kakawanda pavhiki.
  > Mukupera '95 ini ndakaburitsa yangu Webhu dhizaini bhizinesi
  > KUSARIRA PADombo, ASI Dombo RINOGUMBUDZA
  > - Ndira 28 Kushongedza kusingawanzo - shiri dzinoshamisa!
  > Muvhidhiyo isu tinogona zvakare kuona akati wandei bvunzo: kune rumwe rutivi, iyo skrini inopiswa neyakareruka uy

In [8]:
# Length distribution
from collections import Counter
lens = Counter(len(ln.split()) for ln in lines)

total_sents = sum(lens.values())
total_words = sum(k * v for k, v in lens.items())
print(f"sentences:    {total_sents:>12,}")
print(f"words:        {total_words:>12,}")
print(f"avg words/s:  {total_words/total_sents:>12.1f}")
print()
buckets = [(5, 10), (10, 20), (20, 50), (50, 100), (100, 10_000_000)]
for lo, hi in buckets:
    n = sum(v for k, v in lens.items() if lo <= k < hi)
    pct = 100 * n / total_sents
    bar = '█' * int(pct / 2)
    print(f"  {lo:>4}-{hi-1:<5} words: {n:>10,} ({pct:5.1f}%) {bar}")

sentences:       1,372,354
words:          20,835,729
avg words/s:          15.2

     5-9     words:    435,816 ( 31.8%) ███████████████
    10-19    words:    626,010 ( 45.6%) ██████████████████████
    20-49    words:    292,639 ( 21.3%) ██████████
    50-99    words:     16,277 (  1.2%) 
   100-9999999 words:      1,612 (  0.1%) 


## What's next

- `notebooks/03_corpus_stats.ipynb` — turn `corpus_stats.json` into the README's stats table + length-distribution plot.
- Phase 2 — continued pretraining on mT5-small using this corpus.